# Data Collection End-to-End (GTA + Bilby)

This notebook reproduces the **data collection and preprocessing** flow end-to-end with a clear, consistent display format.

What you will see in each step:
- dataset overview (rows, columns, key columns),
- sample rows,
- output file location,
- before/after comparison for cleaning steps.

In [1]:
# Step 0: Setup paths and reusable display helpers.
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import pandas as pd
import polars as pl


def resolve_project_dir() -> Path:
    """Find project root by locating directories that must exist in this repo."""
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "Data").exists() and (candidate / "code").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing both 'Data' and 'code'.")


PROJECT_DIR = resolve_project_dir()
DATA_DIR = PROJECT_DIR / "Data"
PREPROCESS_DIR = PROJECT_DIR / "code" / "data_preprocessing"

GTA_DOWNLOADED_XLSX = DATA_DIR / "China_GTA_Downloaded" / "interventions.xlsx"
GTA_SOURCES_PARQUET = DATA_DIR / "China_GTA_Source" / "interventions_sources.parquet"
GTA_SOURCES_CLEANED_PARQUET = DATA_DIR / "China_GTA_Source" / "interventions_sources_cleaned.parquet"

BILBY_RAW_DIR = DATA_DIR / "Bilby_data_raw"
BILBY_COMBINED_PARQUET = DATA_DIR / "Bilby_data_fixed" / "Bilby_2025_2026_combined.parquet"
BILBY_CLEANED_PARQUET = DATA_DIR / "Bilby_data_fixed" / "Bilby_2025_2026_combined_cleaned.parquet"


def print_section(title: str) -> None:
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)


def detect_date_span(df: pd.DataFrame) -> str:
    """Detect and format dataset date span from common date columns."""
    candidate_columns = [
        "published_date",
        "published_date_cn",
        "published_at",
        "Date Announced",
        "Date Implemented",
        "Date Removed",
    ]

    # Prioritize known date columns used in this project.
    for column in candidate_columns:
        if column in df.columns:
            date_series = pd.to_datetime(df[column], errors="coerce").dropna()
            if not date_series.empty:
                return f"{date_series.min().date()} -> {date_series.max().date()} (column: {column})"

    # Fallback: try all columns to find a valid datetime-like span.
    for column in df.columns:
        date_series = pd.to_datetime(df[column], errors="coerce").dropna()
        if not date_series.empty:
            return f"{date_series.min().date()} -> {date_series.max().date()} (column: {column})"

    return "N/A"


def print_dataset_snapshot(df: pd.DataFrame, title: str, sample_rows: int = 5, output_path: Path | None = None) -> None:
    """Print a consistent, readable summary for any dataframe."""
    print_section(title)
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns)}")
    print(f"Shape: {df.shape}")
    print(f"Date span: {detect_date_span(df)}")
    print(f"Column names: {list(df.columns)}")
    print("\nSample rows:")
    print(df.head(sample_rows))
    if output_path is not None:
        print(f"\nOutput path: {output_path}")


def print_before_after(before_df: pd.DataFrame, after_df: pd.DataFrame, title: str) -> None:
    """Print before/after comparison with row and column changes."""
    before_cols = set(before_df.columns)
    after_cols = set(after_df.columns)
    added_cols = sorted(after_cols - before_cols)
    removed_cols = sorted(before_cols - after_cols)

    print_section(title)
    print(f"Rows before: {len(before_df):,}")
    print(f"Rows after:  {len(after_df):,}")
    print(f"Delta rows:  {len(after_df) - len(before_df):,}")
    print(f"Added columns:   {added_cols if added_cols else 'None'}")
    print(f"Removed columns: {removed_cols if removed_cols else 'None'}")


def run_python_script(script_path: Path, args: list[str] | None = None) -> None:
    """Run existing pipeline script with explicit command display."""
    if args is None:
        args = []
    cmd = [sys.executable, str(script_path), *args]
    print_section("Running script")
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)


print_section("Resolved project paths")
print(f"Project dir: {PROJECT_DIR}")
print(f"GTA downloaded file: {GTA_DOWNLOADED_XLSX}")
print(f"GTA scraped parquet: {GTA_SOURCES_PARQUET}")
print(f"GTA cleaned parquet: {GTA_SOURCES_CLEANED_PARQUET}")
print(f"Bilby raw dir: {BILBY_RAW_DIR}")
print(f"Bilby combined parquet: {BILBY_COMBINED_PARQUET}")
print(f"Bilby cleaned parquet: {BILBY_CLEANED_PARQUET}")


Resolved project paths
Project dir: C:\Users\纯情小火鸡\Desktop\bilby2\bilby2\Bilby code + output
GTA downloaded file: C:\Users\纯情小火鸡\Desktop\bilby2\bilby2\Bilby code + output\Data\China_GTA_Downloaded\interventions.xlsx
GTA scraped parquet: C:\Users\纯情小火鸡\Desktop\bilby2\bilby2\Bilby code + output\Data\China_GTA_Source\interventions_sources.parquet
GTA cleaned parquet: C:\Users\纯情小火鸡\Desktop\bilby2\bilby2\Bilby code + output\Data\China_GTA_Source\interventions_sources_cleaned.parquet
Bilby raw dir: C:\Users\纯情小火鸡\Desktop\bilby2\bilby2\Bilby code + output\Data\Bilby_data_raw
Bilby combined parquet: C:\Users\纯情小火鸡\Desktop\bilby2\bilby2\Bilby code + output\Data\Bilby_data_fixed\Bilby_2025_2026_combined.parquet
Bilby cleaned parquet: C:\Users\纯情小火鸡\Desktop\bilby2\bilby2\Bilby code + output\Data\Bilby_data_fixed\Bilby_2025_2026_combined_cleaned.parquet


## GTA Part - Step 1

### 1) Download GTA dataset in `Data/China_GTA_Downloaded/interventions.xlsx`

Read downloaded GTA data and print consistent preview information.

In [2]:
# GTA Step 1: Read downloaded GTA dataset and print overview.
assert GTA_DOWNLOADED_XLSX.exists(), f"Missing file: {GTA_DOWNLOADED_XLSX}"

gta_downloaded_df = pd.read_excel(GTA_DOWNLOADED_XLSX)
print_dataset_snapshot(
    gta_downloaded_df,
    title="GTA Step 1 - Downloaded interventions.xlsx",
    sample_rows=5,
    output_path=GTA_DOWNLOADED_XLSX,
)


GTA Step 1 - Downloaded interventions.xlsx
Rows: 1,503
Columns: 19
Shape: (1503, 19)
Column names: ['Intervention ID', 'Intervention URL', 'State Act ID', 'State Act URL', 'State Act Title', 'GTA Evaluation', 'Implementing Jurisdictions', 'Implementation Level', 'Eligible Firm', 'Intervention Type', 'Mast Chapter', 'Affected Sectors', 'Affected Products', 'Affected Jurisdictions', 'Is Horizontal', 'Date Announced', 'Date Implemented', 'Date Removed', 'Is In Force']

Sample rows:
   Intervention ID                                  Intervention URL  \
0           122819  https://globaltradealert.org/intervention/122819   
1           131809  https://globaltradealert.org/intervention/131809   
2           131847  https://globaltradealert.org/intervention/131847   
3           132012  https://globaltradealert.org/intervention/132012   
4           132161  https://globaltradealert.org/intervention/132161   

   State Act ID                                     State Act URL  \
0         777

## GTA Part - Step 2

### 2) Scrape source links via `code/data_preprocessing/GTA_Source_Scrape.py`

If you need to rerun scrape, **uncomment** the script-call block in the next cell.

By default, this notebook directly reads and prints:
`Data/China_GTA_Source/interventions_sources.parquet`.

In [3]:
# GTA Step 2: Optional scrape rerun (commented by default), then print scraped parquet info.
gta_scrape_script = PREPROCESS_DIR / "GTA_Source_Scrape.py"

# If you need to rerun scrape, uncomment this block (runtime depends on the size of the downloaded GTA dataset and network conditions):
# run_python_script(gta_scrape_script)

assert GTA_SOURCES_PARQUET.exists(), f"Expected scraped parquet not found: {GTA_SOURCES_PARQUET}"
gta_scraped_df = pd.read_parquet(GTA_SOURCES_PARQUET)
print_dataset_snapshot(
    gta_scraped_df,
    title="GTA Step 2 - Scraped interventions_sources.parquet",
    sample_rows=5,
    output_path=GTA_SOURCES_PARQUET,
)


GTA Step 2 - Scraped interventions_sources.parquet
Rows: 1,518
Columns: 7
Shape: (1518, 7)
Column names: ['main_title', 'source_title', 'source_url', 'site_root_url', 'published_date', 'summary', 'state_act_url']

Sample rows:
                                          main_title  \
0  China: Launch of the CNY 344 billion National ...   
1  China: Launch of the CNY 344 billion National ...   
2  China: Launch of the CNY 344 billion National ...   
3  China: Launch of the CNY 344 billion National ...   
4  China: Government announces sanctions on 5 US ...   

                                        source_title  \
0  National Enterprise Credit Information Publici...   
1  Caixin (28 May 2024). China Piles $47.5 Billio...   
2  Reuters (27 May 2024). China sets up third fun...   
3  Xinhua Finance [新华财经] (27 May 2024). 六大国有银行向大基...   
4  PRC Ministry of Foreign Affairs. "Foreign Mini...   

                                          source_url  \
0  https://www.gsxt.gov.cn/%7B99990D78A871

## GTA Part - Step 3

### 3) Clean GTA data via `code/data_preprocessing/GTA_Bilby_clean.py`

This step calls the shared clean script and relies on GTA clean logic that does:
- `site_root_url` normalization (including `english.` normalization),
- `published_date` parsing/normalization,
- title text cleanup via `clean_gta_title_text`,
- `main_title` enrichment when mapping is available.

In [4]:
# GTA Step 3: Run shared clean script and inspect cleaned GTA parquet.
gta_bilby_clean_script = PREPROCESS_DIR / "GTA_Bilby_clean.py"
run_python_script(
    gta_bilby_clean_script,
    args=[
        "--gta-input", str(GTA_SOURCES_PARQUET),
        "--gta-output", str(GTA_SOURCES_CLEANED_PARQUET),
        "--bilby-input", str(BILBY_COMBINED_PARQUET),
        "--bilby-output", str(BILBY_CLEANED_PARQUET),
    ],
)

assert GTA_SOURCES_CLEANED_PARQUET.exists(), f"Expected cleaned parquet not found: {GTA_SOURCES_CLEANED_PARQUET}"
gta_cleaned_df = pd.read_parquet(GTA_SOURCES_CLEANED_PARQUET)
print_dataset_snapshot(
    gta_cleaned_df,
    title="GTA Step 3 - Cleaned interventions_sources_cleaned.parquet",
    sample_rows=5,
    output_path=GTA_SOURCES_CLEANED_PARQUET,
)


Running script
c:\Users\纯情小火鸡\AppData\Local\Programs\Python\Python314\python.exe C:\Users\纯情小火鸡\Desktop\bilby2\bilby2\Bilby code + output\code\data_preprocessing\GTA_Bilby_clean.py --gta-input C:\Users\纯情小火鸡\Desktop\bilby2\bilby2\Bilby code + output\Data\China_GTA_Source\interventions_sources.parquet --gta-output C:\Users\纯情小火鸡\Desktop\bilby2\bilby2\Bilby code + output\Data\China_GTA_Source\interventions_sources_cleaned.parquet --bilby-input C:\Users\纯情小火鸡\Desktop\bilby2\bilby2\Bilby code + output\Data\Bilby_data_fixed\Bilby_2025_2026_combined.parquet --bilby-output C:\Users\纯情小火鸡\Desktop\bilby2\bilby2\Bilby code + output\Data\Bilby_data_fixed\Bilby_2025_2026_combined_cleaned.parquet

GTA Step 3 - Cleaned interventions_sources_cleaned.parquet
Rows: 964
Columns: 9
Shape: (964, 9)
Column names: ['main_title', 'source_title', 'source_url', 'site_root_url', 'published_date', 'summary', 'state_act_url', 'original_site_root_url', 'site_root_url_normalized']

Sample rows:
                   

In [5]:
# GTA Step 4: Print before/after comparison for GTA cleaning.
print_before_after(
    before_df=gta_scraped_df,
    after_df=gta_cleaned_df,
    title="GTA Step 4 - Before vs After Cleaning",
)


GTA Step 4 - Before vs After Cleaning
Rows before: 1,518
Rows after:  964
Delta rows:  -554
Added columns:   ['original_site_root_url', 'site_root_url_normalized']
Removed columns: None


## Bilby Part - Step 1

### 1) Put Bilby raw files under `Data/Bilby_data_raw`

List raw parquet files and preview one example file.

In [6]:
# Bilby Step 1: Inspect raw directory and preview one raw parquet sample.
assert BILBY_RAW_DIR.exists(), f"Missing directory: {BILBY_RAW_DIR}"

raw_files = sorted(BILBY_RAW_DIR.glob("**/*.parquet"))
assert raw_files, f"No parquet files found under: {BILBY_RAW_DIR}"

print_section("Bilby Step 1 - Raw file inventory")
print(f"Raw parquet count: {len(raw_files)}")
print("Example raw files:")
for path in raw_files[:8]:
    print(f"- {path}")

bilby_raw_sample_df = pd.read_parquet(raw_files[0])
print_dataset_snapshot(
    bilby_raw_sample_df,
    title=f"Bilby Step 1 - Raw sample ({raw_files[0].name})",
    sample_rows=5,
    output_path=raw_files[0],
)


Bilby Step 1 - Raw file inventory
Raw parquet count: 271
Example raw files:
- C:\Users\纯情小火鸡\Desktop\bilby2\bilby2\Bilby code + output\Data\Bilby_data_raw\2025\06\daily_2025-06-01.parquet
- C:\Users\纯情小火鸡\Desktop\bilby2\bilby2\Bilby code + output\Data\Bilby_data_raw\2025\06\daily_2025-06-02.parquet
- C:\Users\纯情小火鸡\Desktop\bilby2\bilby2\Bilby code + output\Data\Bilby_data_raw\2025\06\daily_2025-06-03.parquet
- C:\Users\纯情小火鸡\Desktop\bilby2\bilby2\Bilby code + output\Data\Bilby_data_raw\2025\06\daily_2025-06-04.parquet
- C:\Users\纯情小火鸡\Desktop\bilby2\bilby2\Bilby code + output\Data\Bilby_data_raw\2025\06\daily_2025-06-05.parquet
- C:\Users\纯情小火鸡\Desktop\bilby2\bilby2\Bilby code + output\Data\Bilby_data_raw\2025\06\daily_2025-06-06.parquet
- C:\Users\纯情小火鸡\Desktop\bilby2\bilby2\Bilby code + output\Data\Bilby_data_raw\2025\06\daily_2025-06-07.parquet
- C:\Users\纯情小火鸡\Desktop\bilby2\bilby2\Bilby code + output\Data\Bilby_data_raw\2025\06\daily_2025-06-08.parquet

Bilby Step 1 - Raw sample 

## Bilby Part - Step 2

### 2) Combine Bilby data via `code/data_preprocessing/Bilby_data_combine.py`

Run the combine script, then print dataset overview and sample rows for:
`Data/Bilby_data_fixed/Bilby_2025_2026_combined.parquet`.

In [7]:
# Bilby Step 2: Combine all monthly raw files into one parquet.
bilby_combine_script = PREPROCESS_DIR / "Bilby_data_combine.py"
run_python_script(
    bilby_combine_script,
    args=[
        "--bilby-dir", str(BILBY_RAW_DIR),
        "--output", str(BILBY_COMBINED_PARQUET),
    ],
)

assert BILBY_COMBINED_PARQUET.exists(), f"Expected combined parquet not found: {BILBY_COMBINED_PARQUET}"
bilby_combined_df = pd.read_parquet(BILBY_COMBINED_PARQUET)
print_dataset_snapshot(
    bilby_combined_df,
    title="Bilby Step 2 - Combined Bilby_2025_2026_combined.parquet",
    sample_rows=5,
    output_path=BILBY_COMBINED_PARQUET,
)


Running script
c:\Users\纯情小火鸡\AppData\Local\Programs\Python\Python314\python.exe C:\Users\纯情小火鸡\Desktop\bilby2\bilby2\Bilby code + output\code\data_preprocessing\Bilby_data_combine.py --bilby-dir C:\Users\纯情小火鸡\Desktop\bilby2\bilby2\Bilby code + output\Data\Bilby_data_raw --output C:\Users\纯情小火鸡\Desktop\bilby2\bilby2\Bilby code + output\Data\Bilby_data_fixed\Bilby_2025_2026_combined.parquet

Bilby Step 2 - Combined Bilby_2025_2026_combined.parquet
Rows: 1,163,642
Columns: 16
Shape: (1163642, 16)
Column names: ['uuid', 'branch_id', 'published_at', 'original_article_url', 'article_url_is_proxy', 'article_url', 'site_root_url', 'title', 'subhead', 'summary', 'title_en', 'translated_summary', 'subhead_en', 'news_line', 'newspaper', 'author']

Sample rows:
                                                uuid  branch_id  \
0  4b4b051b-5b31-c8cd-819d-50f48b8b0b6a305443b3-4...   497769.0   
1  885ae848-5269-f7d6-f26d-35d26884a7a4ad1b8bcb-5...  3298635.0   
2  d9d9d549-2b70-4d31-b27c-f28c216be

## Bilby Part - Step 3

### 3) Clean Bilby data via `code/data_preprocessing/GTA_Bilby_clean.py` (Bilby clean section)

The Bilby clean logic includes:
- proxy URL detection and restoring original article URL,
- deriving/normalizing `site_root_url`,
- converting `published_at` to `published_at_cn` and deriving date field,
- keeping deduplicated records by `article_url` from combine stage.

In [8]:
# Bilby Step 3: Run shared clean script and inspect cleaned Bilby parquet.
run_python_script(
    gta_bilby_clean_script,
    args=[
        "--gta-input", str(GTA_SOURCES_PARQUET),
        "--gta-output", str(GTA_SOURCES_CLEANED_PARQUET),
        "--bilby-input", str(BILBY_COMBINED_PARQUET),
        "--bilby-output", str(BILBY_CLEANED_PARQUET),
    ],
)

assert BILBY_CLEANED_PARQUET.exists(), f"Expected cleaned parquet not found: {BILBY_CLEANED_PARQUET}"
bilby_cleaned_df = pd.read_parquet(BILBY_CLEANED_PARQUET)
print_dataset_snapshot(
    bilby_cleaned_df,
    title="Bilby Step 3 - Cleaned Bilby_2025_2026_combined_cleaned.parquet",
    sample_rows=5,
    output_path=BILBY_CLEANED_PARQUET,
)


Running script
c:\Users\纯情小火鸡\AppData\Local\Programs\Python\Python314\python.exe C:\Users\纯情小火鸡\Desktop\bilby2\bilby2\Bilby code + output\code\data_preprocessing\GTA_Bilby_clean.py --gta-input C:\Users\纯情小火鸡\Desktop\bilby2\bilby2\Bilby code + output\Data\China_GTA_Source\interventions_sources.parquet --gta-output C:\Users\纯情小火鸡\Desktop\bilby2\bilby2\Bilby code + output\Data\China_GTA_Source\interventions_sources_cleaned.parquet --bilby-input C:\Users\纯情小火鸡\Desktop\bilby2\bilby2\Bilby code + output\Data\Bilby_data_fixed\Bilby_2025_2026_combined.parquet --bilby-output C:\Users\纯情小火鸡\Desktop\bilby2\bilby2\Bilby code + output\Data\Bilby_data_fixed\Bilby_2025_2026_combined_cleaned.parquet

Bilby Step 3 - Cleaned Bilby_2025_2026_combined_cleaned.parquet
Rows: 1,163,642
Columns: 9
Shape: (1163642, 9)
Column names: ['original_article_url', 'article_url_is_proxy', 'article_url', 'site_root_url', 'title', 'summary', 'title_en', 'original_site_root_url', 'published_date_cn']

Sample rows:
      

In [9]:
# Bilby Step 4: Print before/after comparison for Bilby cleaning.
print_before_after(
    before_df=bilby_combined_df,
    after_df=bilby_cleaned_df,
    title="Bilby Step 4 - Before vs After Cleaning",
)

print_section("Bilby Step 4 - Side-by-side sample rows")
print("Before clean sample:")
print(bilby_combined_df.head(5))
print("\nAfter clean sample:")
print(bilby_cleaned_df.head(5))


Bilby Step 4 - Before vs After Cleaning
Rows before: 1,163,642
Rows after:  1,163,642
Delta rows:  0
Added columns:   ['original_site_root_url', 'published_date_cn']
Removed columns: ['author', 'branch_id', 'news_line', 'newspaper', 'published_at', 'subhead', 'subhead_en', 'translated_summary', 'uuid']

Bilby Step 4 - Side-by-side sample rows
Before clean sample:
                                                uuid  branch_id  \
0  4b4b051b-5b31-c8cd-819d-50f48b8b0b6a305443b3-4...   497769.0   
1  885ae848-5269-f7d6-f26d-35d26884a7a4ad1b8bcb-5...  3298635.0   
2  d9d9d549-2b70-4d31-b27c-f28c216be7dd30edfd06-7...        NaN   
3  0afa0a86-439a-668a-dbf0-1a0a564e12fcba3825b1-3...  7083856.0   
4  ac3ffd37-e550-bca4-c23d-d36f8fd8e2fd6808df3e-e...   332846.0   

               published_at  \
0 2025-05-31 16:00:00+00:00   
1 2025-05-31 16:00:00+00:00   
2 2025-05-31 16:00:00+00:00   
3 2025-05-31 16:00:00+00:00   
4 2025-05-31 16:00:00+00:00   

                                original_ar